# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Using Google Search Console performance metrics and a small set of content attributes, can we build a reason coded system for flagging which pages deserve human review, and does unsupervised clustering of the same features produce groups that align with, or diverge from, that reason coded system?

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


| Item | Detail |
|---|---|
| Dataset release | FlyRank internship warehouse |
| Tables used | dim_content, fact_content_daily_performance |
| Date window | January 2026 to June 2026 |
| Row count | 81,769,613 |

Excluded and why:

Hashed ID columns (content_hash_id, client_hash_id, keyword_hash_id, url_hash_id) were used only for joins and grouping, never as a feature, since a hashed ID carries no real meaning as signal.

GA4 fields, sessions, AI fields, and scroll events were dropped because GA4 coverage was only about 4.2 percent of the training month, too sparse to build anything reliable on.

Search volume, competition, and CPC are real market opportunity signals I considered, but I was capped at 5 features, and visibility, performance, trend, and content investment felt like the core of what defines an archetype. These were cut for scope, not because they don't matter.

Keyword token count was a close call against word count, but token count describes the keyword being targeted more than it describes the content itself.

Clicks were kept underneath CTR as a raw signal, but not counted as a standalone feature.

No client names, raw queries, or account identifying fields appear anywhere in this project.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

One row equals one page, per day, for March 2026. Confirmed no duplicates by checking content_hash_id plus report_date together.

Five features went into the model: impressions, average position, CTR, trend direction, and word count. Each one had to answer one of these:

Does it describe visibility, how visible the page is.
Does it describe performance, how well the page engages people once seen.
Does it describe trend, which way things are moving over time.
Does it describe content investment, an attribute of the content itself feeding into the other three.

Log1p was applied to the features, then a quantile transformer was fit on training data only, never on test data, to avoid leaking test distribution into the transform.

There is no label here. This is fully unsupervised. Any name given to a cluster afterward, like champion or underperformer, is something I assigned after looking at the results, not a real column in the data.

The baseline is a hand coded rule called action score. It flags a page as a declining underperformer when position is worse than 10 and the trend is getting worse. Everything else falls into healthy, weak but stable, or low confidence signal, with that last category built on a Wilson confidence interval on CTR rather than a raw impression count cutoff.

Validation is both time based and grouped. Train is March, test is June, which makes it time based. Within June, I sample by page rather than by row, because random row sampling breaks the run of consecutive days a page needs for trend direction to mean anything.

Three real problems came up during the leakage check. Average position sometimes showed up as exactly 0, which was a data bug that created a fake cluster, fixed by turning those values into nulls. Word count leaked future edit dates into March rows, since word count reflected whatever the page looked like at the time of the pull, not what it looked like in March, and this was corrected before training and validation. Three of the five clustering features also feed directly into the baseline rule, which means agreement between the cluster and the baseline is partly expected rather than fully independent confirmation.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Choosing k

| k | Silhouette train | Inertia |
|---|---|---|
| 2 | 0.6871 | 4,254,114 |
| 3 | 0.6861 | 2,109,838 |
| 4 | 0.5277 | 1,192,798 |
| 5 | 0.4481 | 990,042 |
| 6 | 0.4500 | 838,148 |
| 7 | 0.4304 | 708,687 |
| 8 | 0.4487 | 607,353 |
| 9 | 0.4087 | 537,842 |

Silhouette alone puts k equals 2 slightly ahead of k equals 3, close enough to call it noise rather than a real difference. Inertia settles it. Going from k equals 2 to k equals 3 cuts inertia by roughly half. Going from 3 to 4 barely moves it and silhouette drops hard at the same time, down to 0.5277. That is the elbow, and it lines up with the business case too. Two clusters only gives one dividing line, which just splits high traffic from low traffic and throws away the difference between a healthy page and one that gets seen but never clicked. Three clusters gives three groups a team can actually treat differently.

Checking if this is real structure or a k means artifact

| Model | k | Silhouette train | ARI against k means |
|---|---|---|---|
| K means | 3 | 0.6861 | 1.0000 |
| GMM | 3 | 0.6861 | 1.0000 |
| Agglomerative | 3 | 0.6855 | 1.0000 |

All three methods land on the exact same grouping. Seed stability across six different random seeds also came back at 1.0000. That rules out the idea that this is just k means finding structure because it always finds some structure.

| Data | Silhouette |
|---|---|
| Real data | 0.6861 |
| Column shuffled data | 0.5357 |

Shuffling each column on its own, which keeps every feature's distribution but destroys any real relationship between features, still gives a silhouette of 0.5357. That is lower than the real score, so real relationships between features are contributing something, but 0.5357 is still fairly high on its own, which means part of the score comes from the shape of individual features rather than purely from how they relate to each other.

Cluster profiles on the held out test split

| Cluster | Average position | Trend direction | Impressions | Word count | CTR |
|---|---|---|---|---|---|
| 0 | 13.19 | 0.17 | 1.00 | 3,566.86 | 0.00 |
| 1 | 9.84 | 0.00 | 19.41 | 3,773.37 | 0.00 |
| 2 | 6.13 | -0.03 | 150.54 | 3,452.66 | 1.31 |

Impressions climb steadily from cluster 0 to cluster 2, and CTR only shows up in cluster 2, so visibility is really what's doing the sorting here. Cluster 0 has neither visibility nor clicks, so there is nothing to optimize yet. Cluster 1 has picked up enough visibility to show up in search but isn't converting that into clicks, which points to a messaging or relevance problem rather than a visibility problem. Cluster 2 has both, which confirms that once visibility and positioning line up, the content carries itself.

Cluster agreement against the baseline

| Cluster | Declining underperformer | Healthy | Low confidence signal | Weak but stable |
|---|---|---|---|---|
| 0 | 0.0 percent | 0.0 percent | 100 percent | 0.0 percent |
| 1 | 11.6 percent | 31.7 percent | 48.2 percent | 8.6 percent |
| 2 | 11.3 percent | 66.1 percent | 12.6 percent | 10.0 percent |

Cross split cluster agreement rate came out to 64.66 percent, well above the 33 percent you would expect from chance. About 76 percent of the disagreement is concentrated in cluster 1, while clusters 0 and 2 stay mostly consistent across splits. So the structure holds up overall, but cluster 1 is the one boundary that moves around depending on the split.



## 5. Limitations

*What this work cannot claim.*


Cluster and baseline agreement is partly circular. Three of the five clustering features are also direct inputs into the baseline rule, so agreement between them reflects two related methods more than two independent ones confirming each other.

This is a snapshot, not an early warning system. There is no article text, topic data, or query level data, so the model cannot tell a content quality problem apart from a plain traffic and rank problem, and it cannot catch a decline while it is still happening.

The result rests on one train and test window, March to June, with no repeated splits across other months. It should not be assumed to hold across seasons or over a longer stretch of time without checking that separately.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*


Review declining underperformers first. Both the baseline rule and the clustering agree these are the only pages with an active downward trend past position 10, which makes this the lowest risk place to start.

Hold off on low confidence signals. Their CTR estimates come from too little impression volume to trust, so treating them as urgent risks fixing pages that were never actually broken.

Read the cluster and baseline agreement as related, not independent. Three of the five clustering features are also direct baseline inputs, so when a cluster lines up with a baseline category, that is two methods drawing from overlapping numbers, not two separate signals confirming each other.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


The k selection table, showing silhouette and inertia across k equals 2 through 9.

The 3D PCA scatter plot at k equals 3, with each cluster centroid marked and color matched to its cluster.

The cross algorithm and stability table, comparing k means, GMM, and agglomerative clustering by silhouette and ARI.

The shuffled data comparison table, showing silhouette on real features against silhouette on column shuffled features.

The cluster profile table, showing mean feature values per cluster on the test split.

The cluster agreement table, showing how each cluster's pages break down across the four baseline reason codes.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.